3- Hypothesis Testing

Use a Z-test for proportions and a Chi-Square test to determine statistical significance.

3-1 Setup

In [1]:
import pandas as pd 
import numpy as np 
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv ('../data/marketing_AB_clean.csv')

# Extract key numbers
groups = df.groupby('test group').agg(
    n=('converted', 'count'),
    conversions=('converted', 'sum'),
    rate=('converted', 'mean')
)
print(groups)

                 n  conversions      rate
test group                               
ad          564577        14423  0.025547
psa          23524          420  0.017854


3.2 State the hypotheses

H₀ (null): Conversion rate(ad) = Conversion rate(psa) — the new ad made no difference.

H₁ (alternative): Conversion rate(ad) ≠ Conversion rate(psa) — the new ad changed conversion.

Significance level α = 0.05 (95% confidence)

3.3 Two-proportion Z-test

In [2]:
count = groups['conversions'].values    #[n_ad_converted, n_psa_converted]
nobs = groups['n'].values               #[n_ad_converted, n_psa_converted]

z_stat, p_value = proportions_ztest(count, nobs, alternative= 'two-sided')

print(f"Z-statistic : {z_stat:.4f}")
print(f"P_value)    : {p_value:.6f}")
print()
if p_value < 0.05:
    print("Result: SIGNIFICANT = reject H₀ (p < 0.05)")
    print("The new ad produced a significant difference in conversion rate. ")
else:
    print("Result: NOT SIGNIFICANT- fail to reject H₀ (p > 0.05)")
    print("We cannot conclude the new ad made a real difference.")

Z-statistic : 7.3701
P_value)    : 0.000000

Result: SIGNIFICANT = reject H₀ (p < 0.05)
The new ad produced a significant difference in conversion rate. 


3.4 chi-Square test (cross-check)

In [3]:
# Build contingency table: rows = groups, cols =[converted, not converted]
contingency= pd.crosstab(df['test group'], df['converted'])
print("Contingency table:")
print(contingency)

chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi_Square statistic : {chi2:.4f}")
print(f"Degrees of freedom  : {dof}")
print(f"P_value             : {p_chi2:.6f}")
print()
if p_chi2 < 0.05:
    print("Chi_Square confirms: SIGNIFICANT")
else:
    print("Chi_Square confirms: NOT SIGNIFICANT")


Contingency table:
converted        0      1
test group               
ad          550154  14423
psa          23104    420

Chi_Square statistic : 54.0058
Degrees of freedom  : 1
P_value             : 0.000000

Chi_Square confirms: SIGNIFICANT


In [4]:
import numpy as np

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

rates = groups['rate']
p_ad = rates.get('ad', rates.iloc[0])
p_psa = rates.get('psa', rates.iloc[1])

h = cohens_h(p_ad, p_psa)
print(f"Cohen's h = {h:.4f}")
print("Interpretation:",
      "Small" if abs(h) < 0.2 else ("Medium" if abs(h) < 0.5 else "Large"),
      "effect size")


Cohen's h = 0.0530
Interpretation: Small effect size
